In [ ]:
from ultralytics import YOLO
import cv2
from __future__ import annotations

from pathlib import Path
import json
import random
import cv2
import yaml

In [ ]:
# paths
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
DATASET_DIR = PROJECT_DIR / "data" / "yolo_particle_dataset"

IMAGES_TRAIN_DIR = DATASET_DIR / "images" / "train"
IMAGES_VAL_DIR = DATASET_DIR / "images" / "val"
LABELS_TRAIN_DIR = DATASET_DIR / "labels" / "train"
LABELS_VAL_DIR = DATASET_DIR / "labels" / "val"

for p in [IMAGES_TRAIN_DIR, IMAGES_VAL_DIR, LABELS_TRAIN_DIR, LABELS_VAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ROI_JSON_PATH = DATASET_DIR / "roi_config.json"
YAML_PATH = DATASET_DIR / "data.yaml"

VIDEO_NAMES = ["vid_008", "vid_009", "vid_010"]

# helpers
def find_video(raw_dir: Path, stem_part: str) -> Path:
    matches = [p for p in raw_dir.glob("*.mp4") if stem_part in p.stem]
    if not matches:
        raise FileNotFoundError(f"Could not find video containing '{stem_part}' in {raw_dir}")
    return matches[0]

def get_middle_frame(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    mid = frame_count // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read middle frame from {video_path}")
    return frame, frame_count, mid

def select_roi_once(video_path: Path) -> dict:
    frame, frame_count, mid = get_middle_frame(video_path)
    print(f"\nSelecting ROI for {video_path.name}")
    print(f"Frame count: {frame_count}, middle frame: {mid}")
    x, y, w, h = cv2.selectROI(
        f"Select ROI for {video_path.name} and press ENTER",
        frame,
        fromCenter=False,
        showCrosshair=True
    )
    cv2.destroyAllWindows()

    if w == 0 or h == 0:
        raise ValueError(f"No ROI selected for {video_path.name}")

    return {"x0": int(x), "y0": int(y), "w": int(w), "h": int(h)}

def crop_frame(frame, roi: dict):
    x0, y0, w, h = roi["x0"], roi["y0"], roi["w"], roi["h"]
    return frame[y0:y0+h, x0:x0+w]

def write_data_yaml(yaml_path: Path, dataset_dir: Path):
    data = {
        "path": str(dataset_dir).replace("\\", "/"),
        "train": "images/train",
        "val": "images/val",
        "names": {
            0: "particle"
        }
    }
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

def save_empty_label_file(label_path: Path):
    label_path.parent.mkdir(parents=True, exist_ok=True)
    if not label_path.exists():
        label_path.write_text("", encoding="utf-8")

# ROI
roi_config = {}

for vid_name in VIDEO_NAMES:
    video_path = find_video(RAW_DIR, vid_name)
    roi = select_roi_once(video_path)
    roi_config[vid_name] = roi
    print(f"{vid_name} ROI: {roi}")

with open(ROI_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(roi_config, f, indent=2)

print(f"\nSaved ROI config to: {ROI_JSON_PATH}")

# extract cropped frames
FRAME_STEP = 5
MAX_PER_VIDEO = 120
VAL_RATIO = 0.2
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

all_saved = []

for vid_name in VIDEO_NAMES:
    video_path = find_video(RAW_DIR, vid_name)
    roi = roi_config[vid_name]

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")

    frame_idx = 0
    chosen_frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % FRAME_STEP == 0:
            chosen_frames.append((frame_idx, frame.copy()))

        frame_idx += 1

    cap.release()

    if len(chosen_frames) > MAX_PER_VIDEO:
        chosen_frames = random.sample(chosen_frames, MAX_PER_VIDEO)
        chosen_frames = sorted(chosen_frames, key=lambda x: x[0])

    print(f"{vid_name}: selected {len(chosen_frames)} cropped frames")

    n_val = max(1, int(len(chosen_frames) * VAL_RATIO))
    val_indices = set(random.sample(range(len(chosen_frames)), n_val))

    for i, (fid, frame) in enumerate(chosen_frames):
        cropped = crop_frame(frame, roi)

        img_name = f"{vid_name}_f{fid:06d}.jpg"

        if i in val_indices:
            img_path = IMAGES_VAL_DIR / img_name
            label_path = LABELS_VAL_DIR / img_name.replace(".jpg", ".txt")
            split = "val"
        else:
            img_path = IMAGES_TRAIN_DIR / img_name
            label_path = LABELS_TRAIN_DIR / img_name.replace(".jpg", ".txt")
            split = "train"

        cv2.imwrite(str(img_path), cropped)
        save_empty_label_file(label_path)

        all_saved.append({
            "video": vid_name,
            "frame": fid,
            "split": split,
            "image_path": str(img_path),
            "label_path": str(label_path),
        })

print(f"\nSaved {len(all_saved)} cropped images total.")

# Write data.yaml

write_data_yaml(YAML_PATH, DATASET_DIR)
print(f"Saved YAML to: {YAML_PATH}")

print("\nDone.")
print(f"\nDataset root: {DATASET_DIR}")
print(f"Train images: {IMAGES_TRAIN_DIR}")
print(f"Val images:   {IMAGES_VAL_DIR}")


Selecting ROI for vid_008.mp4
Frame count: 2218, middle frame: 1109
vid_008 ROI: {'x0': 599, 'y0': 427, 'w': 274, 'h': 179}

Selecting ROI for vid_009.mp4
Frame count: 2029, middle frame: 1014
vid_009 ROI: {'x0': 588, 'y0': 421, 'w': 292, 'h': 196}

Selecting ROI for vid_010.mp4
Frame count: 1007, middle frame: 503
vid_010 ROI: {'x0': 589, 'y0': 422, 'w': 300, 'h': 182}

Saved ROI config to: C:\Users\milli\Desktop\capstone_project\data\yolo_particle_dataset\roi_config.json
vid_008: selected 120 cropped frames
vid_009: selected 120 cropped frames
vid_010: selected 120 cropped frames

Saved 360 cropped images total.
Saved YAML to: C:\Users\milli\Desktop\capstone_project\data\yolo_particle_dataset\data.yaml

Done.

Dataset root: C:\Users\milli\Desktop\capstone_project\data\yolo_particle_dataset
Train images: C:\Users\milli\Desktop\capstone_project\data\yolo_particle_dataset\images\train
Val images:   C:\Users\milli\Desktop\capstone_project\data\yolo_particle_dataset\images\val
